# RBER — language-model planner experiments, session A (seeds [0, 1, 2, 3])

**Settings:** Accelerator → **GPU T4** (x1 is enough), Internet → **On**. Then *Save Version → Save & Run All*.
Expected time: about 7.1 h. Output: **`llm_results_sessionA.zip`** (Output tab).

What runs: Qwen2.5-0.5B-Instruct planners trained with VERIF, STEP, EXEC, RBER, RBER-NL rewards
(250 updates × 4 tasks × 8 plans, LR 3e-6), seeds [0, 1, 2, 3]. Protocol: `docs/PREREGISTRATION_final.md` in the repository.
The run is resumable: finished runs are skipped if the notebook is restarted in the same session.

In [ ]:
!pip -q install -U accelerate "transformers>=4.51"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
os.makedirs('rber', exist_ok=True)

In [ ]:
%%writefile rber/__init__.py
"""RBER: Rao-Blackwellized execution rewards for training agentic robot planners."""


In [ ]:
%%writefile rber/domain.py
"""Planning domain: skills with measured success probabilities, tasks built from semantic step categories.

A *skill* is a (Meta-World scripted expert, perturbation level) pair whose success probability was measured
in simulation (``data/skills_metaworld.json``). A *task* is a sequence of K semantic steps (e.g. "open the
container"); every step offers ``M`` valid skills of the matching category plus ``N_INVALID`` skills of other
categories. A *plan* chooses one option per step. Executing a plan runs its skills in order and stops at the
first failure; plan success is the product of per-step successes.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field

import numpy as np

CATEGORIES = {
    "move the object to the target": ["push-v3", "pick-place-v3", "plate-slide-v3", "soccer-v3", "sweep-into-v3",
                                      "coffee-push-v3", "bin-picking-v3"],
    "open the container": ["drawer-open-v3", "window-open-v3", "faucet-open-v3"],
    "press the control": ["button-press-v3", "handle-press-v3", "lever-pull-v3"],
    "close the container": ["drawer-close-v3", "door-close-v3"],
    "use the tool on the part": ["assembly-v3", "hammer-v3"],
    "move the gripper to the marked point": ["reach-v3"],
}
MODES = ("alpha", "beta", "gamma")
N_VALID, N_INVALID = 3, 1
N_OPT = N_VALID + N_INVALID
INVALID = -1
DEFAULT_SKILLS = os.path.join(os.path.dirname(__file__), "..", "..", "data", "skills_metaworld.json")


@dataclass
class Library:
    names: list            # human-readable skill names, e.g. "push[beta]"
    p: np.ndarray          # measured success probability per skill
    category: np.ndarray   # category index per skill
    categories: list       # category names
    ci95: np.ndarray       # Wilson intervals of the measurement


def load_library(path: str = DEFAULT_SKILLS, seed: int = 0) -> Library:
    """Skills from the measurement file. The mode label (alpha/beta/gamma) is assigned by a fixed random
    permutation per base skill, so the name carries no information about the perturbation level."""
    data = json.load(open(path))
    rng = np.random.default_rng(seed)
    cats = list(CATEGORIES)
    by_task: dict = {}
    for s in data["skills"]:
        by_task.setdefault(s["task"], []).append(s)
    names, p, cat, ci = [], [], [], []
    for task, rows in by_task.items():
        rows = sorted(rows, key=lambda r: r["sigma"])
        modes = rng.permutation(MODES)
        c = next(i for i, (k, v) in enumerate(CATEGORIES.items()) if task in v)
        for m, r in zip(modes, rows):
            names.append(f"{task.replace('-v3', '')}[{m}]"); p.append(r["p"]); cat.append(c); ci.append(r["ci95"])
    return Library(names, np.array(p), np.array(cat), cats, np.array(ci))


@dataclass
class Task:
    id: int
    steps: list                 # category index per step
    opts: np.ndarray            # [K, N_OPT] skill index per option
    valid: np.ndarray           # [K, N_OPT] bool
    ref: np.ndarray             # [K] option index of the reliability-blind reference plan (VERIF)


@dataclass
class World:
    lib: Library
    K: int
    train: list
    held: list
    pen: np.ndarray = field(default=None)  # [S, S] multiplicative context factor on the 2nd skill of a pair

    # ---- true execution model -------------------------------------------------------------------------
    def skills(self, t: Task, choice) -> np.ndarray | None:
        """Skill indices of a plan, or None if any chosen option is invalid for its step."""
        c = np.asarray(choice)
        if not t.valid[np.arange(self.K), c].all():
            return None
        return t.opts[np.arange(self.K), c]

    def step_probs(self, sk: np.ndarray) -> np.ndarray:
        q = self.lib.p[sk].astype(float).copy()
        if self.pen is not None:
            q[1:] *= self.pen[sk[:-1], sk[1:]]
        return q

    def success(self, t: Task, choice) -> float:
        """True success probability R(a) of a plan (0 for invalid plans)."""
        sk = self.skills(t, choice)
        return 0.0 if sk is None else float(np.prod(self.step_probs(sk)))

    def optimum(self, t: Task) -> float:
        import itertools
        return max(self.success(t, c) for c in itertools.product(range(N_OPT), repeat=self.K))

    def execute(self, sk: np.ndarray, rng: np.random.Generator):
        """Run skills in order, stop at the first failure. Returns (success, outcomes of attempted steps)."""
        q = self.step_probs(sk)
        out = []
        for k in range(self.K):
            z = bool(rng.random() < q[k]); out.append(z)
            if not z:
                return False, out
        return True, out


def build_world(lib: Library, K: int = 3, n_train: int = 24, n_held: int = 12, seed: int = 0,
                interaction: float = 0.0, interaction_frac: float = 0.25) -> World:
    """Random tasks over distinct step categories. ``interaction`` > 0 multiplies the success of the second
    skill of a random ``interaction_frac`` of ordered skill pairs by (1 - interaction): the per-skill
    independence model used by RBER is then misspecified."""
    rng = np.random.default_rng(seed)
    C, S = len(lib.categories), len(lib.p)
    if K > C:
        raise ValueError(f"K={K} exceeds the number of categories ({C})")

    def make(i):
        steps = list(rng.choice(C, K, replace=False))
        opts = np.zeros((K, N_OPT), int); valid = np.zeros((K, N_OPT), bool)
        for k, c in enumerate(steps):
            good = rng.choice(np.flatnonzero(lib.category == c), N_VALID, replace=False)
            bad = rng.choice(np.flatnonzero(lib.category != c), N_INVALID, replace=False)
            perm = rng.permutation(N_OPT); o = np.r_[good, bad][perm]
            opts[k] = o; valid[k] = perm < N_VALID
        ref = np.array([rng.choice(np.flatnonzero(valid[k])) for k in range(K)])
        return Task(i, steps, opts, valid, ref)

    tasks = [make(i) for i in range(n_train + n_held)]
    pen = None
    if interaction > 0:
        pen = np.where(rng.random((S, S)) < interaction_frac, 1.0 - interaction, 1.0)
    return World(lib, K, tasks[:n_train], tasks[n_train:], pen)


def prompt_text(w: World, t: Task) -> str:
    """Natural-language task description shown to the language-model planner."""
    cats = w.lib.categories
    order = ["first"] + ["then"] * (w.K - 2) + ["finally"] if w.K > 1 else ["first"]
    head = "Robot task: " + ", ".join(f"{o} {cats[c]}" for o, c in zip(order, t.steps)) + "."
    lines = [head, "Choose one skill for each step from its options."]
    for k in range(w.K):
        opts = "  ".join(f"{'ABCD'[j]}) {w.lib.names[s]}" for j, s in enumerate(t.opts[k]))
        lines.append(f"Step {k + 1} ({cats[t.steps[k]]}): {opts}")
    return "\n".join(lines)


In [ ]:
%%writefile rber/rewards.py
"""Reward signals for training a planner, all computed from the same sampled plans.

VERIF   offline verification: plan valid and agreement with a reliability-blind reference plan (no execution)
STEP    execution, dense progress: fraction of steps completed before the first failure
EXEC    execution, binary success of the plan
RBER    execution, Rao-Blackwellized execution reward: product of posterior-mean skill success probabilities.
        The posterior is shared by all tasks and is *lagged*: the reward of a batch uses the posterior
        before that batch's outcomes are added (Section IV; makes the reward independent of the sample's own
        outcome given the history).
RBER-L  RBER with one posterior per task (no sharing across tasks)                 [ablation]
RBER-NL RBER whose posterior already includes the current batch (non-lagged; used in our pilot) [ablation]
RB*     oracle Rao-Blackwellization with the true success probability R(a)     [upper reference]
"""
from __future__ import annotations

import numpy as np

METHODS = ("VERIF", "STEP", "EXEC", "RBER", "RBER-L", "RBER-NL", "RB*")
EXECUTES = {"STEP", "EXEC", "RBER", "RBER-L", "RBER-NL", "RB*"}


class SkillPosterior:
    """Independent Beta(a0, b0) posteriors over skill success probabilities, optionally one set per task."""

    def __init__(self, n_skills: int, n_groups: int = 1, a0: float = 1.0, b0: float = 1.0):
        self.a = np.full((n_groups, n_skills), a0, float)
        self.b = np.full((n_groups, n_skills), b0, float)

    def mean(self, g: int = 0) -> np.ndarray:
        return self.a[g] / (self.a[g] + self.b[g])

    def count(self, g: int = 0) -> np.ndarray:
        return self.a[g] + self.b[g]

    def update(self, sk: np.ndarray, outcomes: list, g: int = 0):
        """Censored observation: steps up to and including the first failure."""
        for s, z in zip(sk, outcomes):
            self.a[g, s] += z; self.b[g, s] += 1 - z


def batch_rewards(method: str, world, task, choices: np.ndarray, rng: np.random.Generator,
                  post: SkillPosterior | None = None, group: int = 0, reward_post: SkillPosterior | None = None):
    """Rewards for G sampled plans of one task. Returns (rewards[G], executions used).

    ``post`` is updated with the censored outcomes of the executed plans. For RBER/RBER-L the reward uses
    ``reward_post`` if given (a snapshot taken before the whole batch, used when a batch contains several
    tasks), otherwise ``post`` before this call's updates."""
    G = len(choices)
    r = np.zeros(G)
    plans = [world.skills(task, c) for c in choices]
    if method == "VERIF":
        for i, (c, sk) in enumerate(zip(choices, plans)):
            r[i] = 0.0 if sk is None else float(np.mean(np.asarray(c) == task.ref))
        return r, 0
    if method in ("RBER", "RBER-L"):  # lagged: reward from the posterior *before* this batch
        m = (reward_post if reward_post is not None else post).mean(group)
        for i, sk in enumerate(plans):
            r[i] = 0.0 if sk is None else float(np.prod(m[sk]))
    if method == "RB*":
        for i, c in enumerate(choices):
            r[i] = world.success(task, c)
    n_exec = 0
    for i, sk in enumerate(plans):
        if sk is None:
            continue                      # an invalid plan fails at once and costs no execution
        n_exec += 1
        ok, out = world.execute(sk, rng)
        if post is not None:
            post.update(sk[:len(out)], out, group)
        if method == "EXEC":
            r[i] = float(ok)
        elif method == "STEP":
            r[i] = sum(out[:len(out) - (not ok)]) / world.K if not ok else 1.0
    if method == "RBER-NL":  # posterior including this batch
        m = post.mean(group)
        for i, sk in enumerate(plans):
            r[i] = 0.0 if sk is None else float(np.prod(m[sk]))
    return r, n_exec


def group_advantage(r: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Group-normalized advantage (GRPO-style); zero when all rewards in the group are equal."""
    return (r - r.mean()) / (r.std() + eps)


In [ ]:
%%writefile rber/tabular.py
"""Tabular softmax planner (one logit table per task and step) trained with group-baseline policy gradients.

The update is REINFORCE with a group-normalized baseline, one on-policy step per batch (identical to GRPO
with a single optimisation step, where the importance ratio is 1 and clipping is inactive).
"""
from __future__ import annotations

import numpy as np

from .domain import N_OPT
from .rewards import SkillPosterior, batch_rewards, group_advantage


def softmax(x):
    e = np.exp(x - x.max(-1, keepdims=True)); return e / e.sum(-1, keepdims=True)


def greedy_value(world, L, tasks=None):
    tasks = world.train if tasks is None else tasks
    return float(np.mean([world.success(t, L[i].argmax(-1)) for i, t in enumerate(tasks)]))


def train(world, method: str, lr: float, seed: int, n_updates: int = 2000, G: int = 8, eval_every: int = 50,
          prior=(1.0, 1.0)):
    """Returns a dict with learning curves (update, executions, normalized greedy success)."""
    rng = np.random.default_rng(seed)
    exec_rng = np.random.default_rng(seed + 10_000)      # separate stream for execution outcomes
    T, K = len(world.train), world.K
    L = np.zeros((T, K, N_OPT))
    n_groups = T if method == "RBER-L" else 1
    post = SkillPosterior(len(world.lib.p), n_groups, *prior) if method.startswith("RBER") else None
    opt = np.mean([world.optimum(t) for t in world.train])
    curve = [(0, 0, greedy_value(world, L) / opt)]
    execs = 0
    for u in range(1, n_updates + 1):
        i = int(rng.integers(T)); t = world.train[i]
        P = softmax(L[i])                                                  # [K, N_OPT]
        u01 = rng.random((G, K, 1))
        ch = (u01 > np.cumsum(P, -1)[None]).sum(-1)                        # inverse-CDF sampling [G, K]
        ch = np.minimum(ch, N_OPT - 1)
        r, n = batch_rewards(method, world, t, ch, exec_rng, post, group=i if method == "RBER-L" else 0)
        execs += n
        A = group_advantage(r)
        onehot = np.eye(N_OPT)[ch]                                         # [G, K, N_OPT]
        L[i] += lr * (A[:, None, None] * (onehot - P[None])).mean(0)
        if u % eval_every == 0:
            curve.append((u, execs, greedy_value(world, L) / opt))
    return dict(method=method, lr=lr, seed=seed, curve=curve, optimum=opt,
                final=curve[-1][2], held=None)


In [ ]:
%%writefile rber/extension.py
"""Extension study (docs/PREREGISTRATION_extension.md): leave-one-out and contextual RBER, and stronger baselines.

Methods trained by :func:`train_ext` (tabular planner, identical random streams to ``rber.tabular.train``):

RBER-LOO   product of posterior means, posterior = history + all *other* plans of the batch (own outcomes excluded)
RBER-C     first-order contextual model m_{s|c} (c = preceding skill or START) shrunk to m_s, leave-one-out
EXEC-RLOO  binary reward, leave-one-out mean baseline, no std normalization
EXEC-PPO   binary reward, group-normalized advantage, 4 clipped epochs per batch (eps = 0.2)
LSP-lin    learned success predictor prod_k sigmoid(theta_{s_k}) fitted on final outcomes only (replay + Adam)
LSP-mlp    learned success predictor, MLP on per-step one-hot skills, fitted on final outcomes only
TS-plan    Thompson sampling over plans with the shared Beta posterior (model-based reference, not a policy)
Also accepts the original methods (EXEC, STEP, RBER, ...) and then reproduces ``rber.tabular.train`` exactly.
"""
from __future__ import annotations

import itertools

import numpy as np

from .domain import N_OPT
from .rewards import SkillPosterior, batch_rewards, group_advantage
from .tabular import greedy_value, softmax

EXT_METHODS = ("RBER-LOO", "RBER-C", "EXEC-RLOO", "EXEC-PPO", "LSP-lin", "LSP-mlp", "TS-plan")


# ----------------------------------------------------------------------------------------------- posteriors
class ContextPosterior:
    """Marginal Beta(a0, b0) counts per skill plus success/attempt counts per (context, skill), where the context
    is the preceding skill, or a START token (index n_skills) for the first step of a plan."""

    def __init__(self, n_skills: int, a0: float = 1.0, b0: float = 1.0, kappa: float = 2.0):
        self.S = n_skills
        self.a = np.full(n_skills, a0); self.b = np.full(n_skills, b0)
        self.ps = np.zeros((n_skills + 1, n_skills)); self.pn = np.zeros((n_skills + 1, n_skills))
        self.kappa = kappa

    def _ctx(self, sk, k):
        return self.S if k == 0 else sk[k - 1]

    def add(self, sk, out, sign=1.0):
        for k, z in enumerate(out):
            s = sk[k]; self.a[s] += sign * z; self.b[s] += sign * (1 - z)
            c = self._ctx(sk, k); self.ps[c, s] += sign * z; self.pn[c, s] += sign

    def plan_value(self, sk, own=None, contextual=True):
        """Posterior-predictive success of a plan; `own` = (skills, outcomes) of this plan, removed first
        (leave-one-out). Contextual: m_{s|c} = (kappa m_s + S_{c,s}) / (kappa + N_{c,s})."""
        a, b = self.a, self.b
        if own is not None:
            osk, oout = own
            a = a.copy(); b = b.copy()
            for k, z in enumerate(oout):
                a[osk[k]] -= z; b[osk[k]] -= 1 - z
        m = a[sk] / (a[sk] + b[sk])
        if not contextual:
            return float(np.prod(m))
        v = 1.0
        for k in range(len(sk)):
            c = self._ctx(sk, k); S, N = self.ps[c, sk[k]], self.pn[c, sk[k]]
            if own is not None and k < len(own[1]):          # own attempt of step k used exactly this (c, s)
                S -= own[1][k]; N -= 1
            v *= (self.kappa * m[k] + S) / (self.kappa + N)
        return float(v)


def loo_batch_multi(method, world, tasks, choices_list, rng, post: ContextPosterior):
    """Execute every valid plan of a batch (several tasks), add all outcomes to the posterior, then reward each
    plan with the posterior that excludes only its own outcomes (leave-one-out over the whole batch)."""
    plans = [[world.skills(t, c) for c in ch] for t, ch in zip(tasks, choices_list)]
    obs, n = [], 0
    for P in plans:
        o_t = []
        for sk in P:
            if sk is None:
                o_t.append(None); continue
            ok, out = world.execute(sk, rng); n += 1
            o = (sk[:len(out)], [int(z) for z in out]); post.add(*o); o_t.append(o)
        obs.append(o_t)
    rewards = []
    for P, o_t in zip(plans, obs):
        r = np.zeros(len(P))
        for i, sk in enumerate(P):
            if sk is not None:
                r[i] = post.plan_value(sk, own=o_t[i], contextual=(method == "RBER-C"))
        rewards.append(r)
    return rewards, n


def loo_batch(method, world, task, choices, rng, post: ContextPosterior):
    r, n = loo_batch_multi(method, world, [task], [choices], rng, post)
    return r[0], n


# ----------------------------------------------------------------------------------------------- learned predictors
class Adam:
    def __init__(self, params, lr):
        self.p, self.lr, self.t = params, lr, 0
        self.m = [np.zeros_like(x) for x in params]; self.v = [np.zeros_like(x) for x in params]

    def step(self, grads):
        self.t += 1
        for x, g, m, v in zip(self.p, grads, self.m, self.v):
            m *= 0.9; m += 0.1 * g; v *= 0.999; v += 0.001 * g * g
            x += self.lr * (m / (1 - 0.9 ** self.t)) / (np.sqrt(v / (1 - 0.999 ** self.t)) + 1e-8)  # ascent


class SuccessPredictor:
    """P(Y=1 | plan) learned from final binary outcomes only (no step outcomes)."""

    def __init__(self, kind, n_skills, K, lr, rng, hidden=64):
        self.kind, self.S, self.K, self.rng = kind, n_skills, K, rng
        if kind == "lin":
            self.theta = np.zeros(n_skills); self.opt = Adam([self.theta], lr)
        else:
            d = K * n_skills
            self.W1 = rng.normal(0, 1 / np.sqrt(K), (hidden, d)); self.b1 = np.zeros(hidden)
            self.w2 = rng.normal(0, 1 / np.sqrt(hidden), hidden); self.b2 = np.zeros(1)
            self.opt = Adam([self.W1, self.b1, self.w2, self.b2], lr)
        self.X, self.Y = [], []

    def _x(self, SK):
        X = np.zeros((len(SK), self.K * self.S))
        for i, sk in enumerate(SK):
            X[i, np.arange(self.K) * self.S + sk] = 1
        return X

    def predict(self, SK):
        SK = np.asarray(SK)
        if self.kind == "lin":
            return np.prod(1 / (1 + np.exp(-self.theta[SK])), axis=1)
        h = np.tanh(self._x(SK) @ self.W1.T + self.b1)
        return 1 / (1 + np.exp(-(h @ self.w2 + self.b2[0])))

    def add(self, sk, y):
        self.X.append(np.asarray(sk)); self.Y.append(float(y))

    def fit(self, steps=4, batch=64):
        if not self.X:
            return
        for _ in range(steps):
            idx = self.rng.integers(0, len(self.X), batch)
            SK = np.stack([self.X[i] for i in idx]); y = np.array([self.Y[i] for i in idx])
            if self.kind == "lin":
                sg = 1 / (1 + np.exp(-self.theta[SK]))                       # [B, K]
                P = np.clip(np.prod(sg, 1), 1e-9, 1 - 1e-6)
                coef = np.where(y == 1, 1.0, -P / (1 - P))                  # d loglik / d log P
                g = np.zeros_like(self.theta)
                np.add.at(g, SK, coef[:, None] * (1 - sg))
                self.opt.step([g / batch])
            else:
                X = self._x(SK); h = np.tanh(X @ self.W1.T + self.b1)
                p = 1 / (1 + np.exp(-(h @ self.w2 + self.b2[0])))
                d = (y - p) / batch                                         # d loglik / d logit
                gw2 = h.T @ d; gb2 = np.array([d.sum()])
                dh = np.outer(d, self.w2) * (1 - h ** 2)
                self.opt.step([dh.T @ X, dh.sum(0), gw2, gb2])


# ----------------------------------------------------------------------------------------------- training loop
def _valid_plans(world, t):
    rows = []
    for c in itertools.product(range(N_OPT), repeat=world.K):
        sk = world.skills(t, np.array(c))
        if sk is not None:
            rows.append((np.array(c), sk))
    return rows


def train_ext(world, method, lr, seed, n_updates=3000, G=8, eval_every=50, prior=(1.0, 1.0), model_lr=0.05,
              kappa=2.0, ppo_epochs=4, clip=0.2):
    rng = np.random.default_rng(seed)
    exec_rng = np.random.default_rng(seed + 10_000)
    aux_rng = np.random.default_rng(seed + 20_000)        # predictor minibatches / Thompson samples
    T, K, S = len(world.train), world.K, len(world.lib.p)
    L = np.zeros((T, K, N_OPT))
    opt = np.mean([world.optimum(t) for t in world.train])
    ctx = ContextPosterior(S, *prior, kappa=kappa) if method in ("RBER-LOO", "RBER-C", "TS-plan") else None
    lsp = SuccessPredictor(method[4:], S, K, model_lr, aux_rng) if method.startswith("LSP") else None
    base_method = {"EXEC-RLOO": "EXEC", "EXEC-PPO": "EXEC"}.get(method, method)
    post = SkillPosterior(S, T if method == "RBER-L" else 1, *prior) if base_method.startswith("RBER") and ctx is None else None
    vp = [_valid_plans(world, t) for t in world.train] if method == "TS-plan" else None

    def evaluate():
        if method == "TS-plan":
            m = ctx.a / (ctx.a + ctx.b)
            return float(np.mean([world.success(t, max(vp[i], key=lambda r: np.prod(m[r[1]]))[0])
                                  for i, t in enumerate(world.train)])) / opt
        return greedy_value(world, L) / opt

    curve = [(0, 0, evaluate())]; execs = 0
    for u in range(1, n_updates + 1):
        i = int(rng.integers(T)); t = world.train[i]
        P = softmax(L[i])
        u01 = rng.random((G, K, 1))
        ch = np.minimum((u01 > np.cumsum(P, -1)[None]).sum(-1), N_OPT - 1)
        if method == "TS-plan":
            for _ in range(G):
                th = aux_rng.beta(ctx.a, ctx.b)
                c, sk = max(vp[i], key=lambda r: np.prod(th[r[1]]))
                ok, out = world.execute(sk, exec_rng); execs += 1
                ctx.add(sk[:len(out)], [int(z) for z in out])
        else:
            if method in ("RBER-LOO", "RBER-C"):
                r, n = loo_batch(method, world, t, ch, exec_rng, ctx)
            elif lsp is not None:
                plans = [world.skills(t, c) for c in ch]
                r = np.zeros(G); n = 0
                valid = [j for j, sk in enumerate(plans) if sk is not None]
                if valid:
                    r[valid] = lsp.predict([plans[j] for j in valid])     # lagged: model before this batch
                for j in valid:
                    ok, out = world.execute(plans[j], exec_rng); n += 1; lsp.add(plans[j], ok)
                lsp.fit()
            else:
                r, n = batch_rewards(base_method, world, t, ch, exec_rng, post, group=i if method == "RBER-L" else 0)
            execs += n
            onehot = np.eye(N_OPT)[ch]
            if method == "EXEC-RLOO":
                A = r - (r.sum() - r) / (G - 1)
                L[i] += lr * (A[:, None, None] * (onehot - P[None])).mean(0)
            elif method == "EXEC-PPO":
                A = group_advantage(r); lp_old = np.log(P[np.arange(K), ch]).sum(1)
                for _ in range(ppo_epochs):
                    Pn = softmax(L[i]); ratio = np.exp(np.log(Pn[np.arange(K), ch]).sum(1) - lp_old)
                    act = ~(((A > 0) & (ratio > 1 + clip)) | ((A < 0) & (ratio < 1 - clip)))
                    L[i] += lr * ((act * A * ratio)[:, None, None] * (onehot - Pn[None])).mean(0)
            else:
                A = group_advantage(r)
                L[i] += lr * (A[:, None, None] * (onehot - P[None])).mean(0)
        if u % eval_every == 0:
            curve.append((u, execs, evaluate()))
    return dict(method=method, lr=lr, seed=seed, curve=curve, optimum=opt, final=curve[-1][2], held=None)


In [ ]:
%%writefile rber/llm.py
"""Language-model planner (Qwen2.5-0.5B-Instruct) trained with group-baseline policy gradients.

Policy: for step k of a task, the model scores every candidate skill *name* by its length-normalized
log-likelihood after the prompt and the prefix "Step k:"; the policy is the softmax over the candidates
(SayCan-style scoring), independently per step. Rewards come from ``rber.rewards`` (identical code to the
tabular experiments). Evaluation: true success of the greedy plan on training and held-out tasks.
"""
from __future__ import annotations

import copy
import gc
import json
import os
import time

import numpy as np
import torch

from .domain import N_OPT, prompt_text
from .extension import ContextPosterior, loo_batch_multi
from .rewards import SkillPosterior, batch_rewards, group_advantage


class LMPolicy:
    def __init__(self, model_id: str, lr: float, device: str | None = None, dtype=torch.float32, micro: int = 24):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_id)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        try:
            m = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype)
        except TypeError:  # older transformers
            m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
        self.model = m.to(dtype).to(self.dev)  # fp32 weights: T4 GPUs have no native bf16
        if self.dev == "cuda":
            self.model.gradient_checkpointing_enable()
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=0.0)
        self.micro = micro
        self._cache = {}

    def enc(self, x):
        return self.tok.encode(x, add_special_tokens=False)

    def close(self):
        """Free GPU memory (model, gradients, optimizer state) immediately."""
        self.opt.zero_grad(set_to_none=True)
        del self.opt, self.model
        self._cache.clear(); gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ---- scoring --------------------------------------------------------------------------------------
    def _task_seqs(self, w, t):
        key = (id(w), t.id)
        if key not in self._cache:
            chat = self.tok.apply_chat_template([{"role": "user", "content": prompt_text(w, t)}], tokenize=False,
                                                add_generation_prompt=True)
            p = self.enc(chat); out = []
            for k in range(w.K):
                pre = p + self.enc(f"Step {k + 1}:")
                for j in range(N_OPT):
                    c = self.enc(" " + w.lib.names[t.opts[k, j]])
                    out.append((pre + c, len(pre), c))
            self._cache[key] = out
        return self._cache[key]

    def _seq_logp(self, items):
        """Length-normalized log-likelihood of each candidate name (full-vocabulary softmax only at the
        candidate positions; the full logit tensor is never materialized)."""
        n = max(len(x[0]) for x in items); pad = self.tok.pad_token_id
        ids = torch.full((len(items), n), pad, dtype=torch.long); att = torch.zeros_like(ids); off = []
        for i, (x, _, _) in enumerate(items):
            ids[i, n - len(x):] = torch.tensor(x); att[i, n - len(x):] = 1; off.append(n - len(x))
        ids, att = ids.to(self.dev), att.to(self.dev)
        pos = (att.cumsum(1) - 1).clamp(min=0)
        h = self.model.base_model(input_ids=ids, attention_mask=att, position_ids=pos).last_hidden_state
        head = self.model.get_output_embeddings()
        rows, cols, tgt, owner = [], [], [], []
        for i, (x, st, c) in enumerate(items):
            for q, tokid in enumerate(c):
                rows.append(i); cols.append(off[i] + st + q - 1); tgt.append(tokid); owner.append(i)
        hs = h[torch.tensor(rows, device=self.dev), torch.tensor(cols, device=self.dev)]
        logits = (hs @ head.weight.T).float()
        if getattr(head, "bias", None) is not None:
            logits = logits + head.bias.float()
        lp = torch.log_softmax(logits, -1).gather(1, torch.tensor(tgt, device=self.dev)[:, None]).squeeze(1)
        tot = torch.zeros(len(items), device=self.dev).index_add(0, torch.tensor(owner, device=self.dev), lp)
        return tot / torch.tensor([len(c) for _, _, c in items], dtype=torch.float32, device=self.dev)

    def dist(self, w, tasks):
        """log pi[t, k, j] (differentiable)."""
        items = [it for t in tasks for it in self._task_seqs(w, t)]
        parts = [self._seq_logp(items[i:i + self.micro]) for i in range(0, len(items), self.micro)]
        return torch.log_softmax(torch.cat(parts).view(len(tasks), w.K, N_OPT), -1)

    def fast_dist(self, w, tasks):
        """No-grad policy for sampling/evaluation; fp16 autocast on GPU with an fp32 fallback."""
        self.model.eval()
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=self.dev == "cuda"):
            lp = self.dist(w, tasks).float()
        if not torch.isfinite(lp).all():
            with torch.no_grad():
                lp = self.dist(w, tasks)
        return lp

    def sample(self, w, tasks, n, rng, greedy=False):
        p = self.fast_dist(w, tasks).exp().cpu().numpy().astype(np.float64)
        ch = np.zeros((len(tasks), n, w.K), int)
        for ti in range(len(tasks)):
            for k in range(w.K):
                pk = p[ti, k] / p[ti, k].sum()
                ch[ti, :, k] = pk.argmax() if greedy else rng.choice(N_OPT, size=n, p=pk)
        return ch

    def update(self, w, tasks, choices, adv):
        """One on-policy step: loss = -(1/N) sum_g adv_g sum_k log pi(choice_gk). Backward task by task."""
        self.model.train(); self.opt.zero_grad(set_to_none=True)
        N = choices.shape[0] * choices.shape[1]
        for ti, t in enumerate(tasks):
            a = torch.tensor(adv[ti], dtype=torch.float32, device=self.dev)
            if a.abs().sum() == 0:
                continue
            lp = self.dist(w, [t])[0]
            ch = torch.tensor(choices[ti], device=self.dev)
            lpc = lp[torch.arange(w.K, device=self.dev)[None, :], ch].sum(1)
            (-(a * lpc).sum() / N).backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.opt.step()


def evaluate(pol, w, tasks, rng):
    ch = pol.sample(w, tasks, 1, rng, greedy=True)[:, 0]
    return float(np.mean([w.success(t, c) for t, c in zip(tasks, ch)]))


def entropy(pol, w, tasks):
    p = pol.fast_dist(w, tasks).exp().cpu().numpy()
    return float((-(p * np.log(p + 1e-12)).sum(-1)).mean())


def train(w, method, seed, out_dir, model_id="Qwen/Qwen2.5-0.5B-Instruct", lr=3e-6, n_updates=250,
          tasks_per_update=4, G=8, eval_every=25, log=print):
    """Resumable at the run level: a finished run (done=True) is skipped."""
    path = os.path.join(out_dir, f"llm_{method}_s{seed}.json")
    if os.path.exists(path) and json.load(open(path)).get("done"):
        log(f"{method} s{seed}: already done"); return path
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed); exec_rng = np.random.default_rng(seed + 10_000)
    pol = LMPolicy(model_id, lr)
    ctx = ContextPosterior(len(w.lib.p)) if method in ("RBER-LOO", "RBER-C") else None
    post = SkillPosterior(len(w.lib.p)) if method.startswith("RBER") and ctx is None else None
    opt_tr = float(np.mean([w.optimum(t) for t in w.train])); opt_he = float(np.mean([w.optimum(t) for t in w.held]))
    hist, execs, t0 = [], 0, time.time()

    def do_eval(u):
        tr = evaluate(pol, w, w.train, rng); he = evaluate(pol, w, w.held, rng); ent = entropy(pol, w, w.train)
        hist.append(dict(update=u, execs=execs, train=tr, held=he, train_norm=tr / opt_tr, held_norm=he / opt_he,
                         entropy=ent, time=time.time() - t0))
        log(f"[{method} s{seed}] upd {u:3d} execs {execs:5d} | train {tr:.3f}/{opt_tr:.3f} held {he:.3f}/{opt_he:.3f}"
            f" entropy {ent:.2f} {time.time() - t0:.0f}s")
        json.dump(dict(method=method, seed=seed, lr=lr, hist=hist, opt_train=opt_tr, opt_held=opt_he, done=False),
                  open(path, "w"))

    do_eval(0)
    for u in range(1, n_updates + 1):
        idx = rng.choice(len(w.train), tasks_per_update, replace=False)
        tasks = [w.train[i] for i in idx]
        ch = pol.sample(w, tasks, G, rng)
        adv = np.zeros((tasks_per_update, G))
        if ctx is not None:                                          # leave-one-out over the whole batch
            rs, n = loo_batch_multi(method, w, tasks, list(ch), exec_rng, ctx); execs += n
            for ti, r in enumerate(rs):
                adv[ti] = group_advantage(r)
        else:
            snap = copy.deepcopy(post) if post is not None else None   # lag: posterior before the whole batch
            for ti, t in enumerate(tasks):
                r, n = batch_rewards(method, w, t, ch[ti], exec_rng, post, reward_post=snap)
                execs += n; adv[ti] = group_advantage(r)
        pol.update(w, tasks, ch, adv)
        if u % eval_every == 0 or u == n_updates:
            do_eval(u)
    json.dump(dict(method=method, seed=seed, lr=lr, hist=hist, opt_train=opt_tr, opt_held=opt_he, done=True),
              open(path, "w"))
    pol.close(); del pol
    return path


In [ ]:
%%writefile skills_metaworld.json
{
 "meta": {
  "metaworld_version": "3.x",
  "mujoco_version": "3.3.0",
  "horizon": 200,
  "levels": [
   0.3,
   0.8,
   1.5
  ],
  "rollouts": 100,
  "seconds": 410
 },
 "skills": [
  {
   "task": "bin-picking-v3",
   "sigma": 0.3,
   "successes": 85,
   "rollouts": 100,
   "p": 0.85,
   "ci95": [
    0.7671644033458782,
    0.9069401475158245
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0
   ]
  },
  {
   "task": "bin-picking-v3",
   "sigma": 0.8,
   "successes": 16,
   "rollouts": 100,
   "p": 0.16,
   "ci95": [
    0.10095288451828571,
    0.2442026946446318
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0
   ]
  },
  {
   "task": "bin-picking-v3",
   "sigma": 1.5,
   "successes": 2,
   "rollouts": 100,
   "p": 0.02,
   "ci95": [
    0.0055019675006616475,
    0.07001179131757478
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "assembly-v3",
   "sigma": 0.3,
   "successes": 26,
   "rollouts": 100,
   "p": 0.26,
   "ci95": [
    0.18404698413091963,
    0.3537098952781987
   ],
   "outcomes": [
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1
   ]
  },
  {
   "task": "assembly-v3",
   "sigma": 0.8,
   "successes": 4,
   "rollouts": 100,
   "p": 0.04,
   "ci95": [
    0.01566330388275538,
    0.09837071498472119
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "assembly-v3",
   "sigma": 1.5,
   "successes": 0,
   "rollouts": 100,
   "p": 0.0,
   "ci95": [
    0.0,
    0.03699349876899627
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "window-open-v3",
   "sigma": 0.3,
   "successes": 98,
   "rollouts": 100,
   "p": 0.98,
   "ci95": [
    0.9299882086824253,
    0.9944980324993383
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "window-open-v3",
   "sigma": 0.8,
   "successes": 98,
   "rollouts": 100,
   "p": 0.98,
   "ci95": [
    0.9299882086824253,
    0.9944980324993383
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "window-open-v3",
   "sigma": 1.5,
   "successes": 81,
   "rollouts": 100,
   "p": 0.81,
   "ci95": [
    0.7222115454423077,
    0.8748524853209148
   ],
   "outcomes": [
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "button-press-v3",
   "sigma": 0.3,
   "successes": 100,
   "rollouts": 100,
   "p": 1.0,
   "ci95": [
    0.9630065012310038,
    1.0
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "button-press-v3",
   "sigma": 0.8,
   "successes": 98,
   "rollouts": 100,
   "p": 0.98,
   "ci95": [
    0.9299882086824253,
    0.9944980324993383
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "button-press-v3",
   "sigma": 1.5,
   "successes": 77,
   "rollouts": 100,
   "p": 0.77,
   "ci95": [
    0.6784561689907251,
    0.841567341674017
   ],
   "outcomes": [
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "hammer-v3",
   "sigma": 0.3,
   "successes": 66,
   "rollouts": 100,
   "p": 0.66,
   "ci95": [
    0.56277728776086,
    0.7453847926330613
   ],
   "outcomes": [
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "hammer-v3",
   "sigma": 0.8,
   "successes": 16,
   "rollouts": 100,
   "p": 0.16,
   "ci95": [
    0.10095288451828571,
    0.2442026946446318
   ],
   "outcomes": [
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "hammer-v3",
   "sigma": 1.5,
   "successes": 5,
   "rollouts": 100,
   "p": 0.05,
   "ci95": [
    0.021543679017992173,
    0.1117504698741045
   ],
   "outcomes": [
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "soccer-v3",
   "sigma": 0.3,
   "successes": 82,
   "rollouts": 100,
   "p": 0.82,
   "ci95": [
    0.7333264153744679,
    0.8829977454133746
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    1
   ]
  },
  {
   "task": "soccer-v3",
   "sigma": 0.8,
   "successes": 60,
   "rollouts": 100,
   "p": 0.6,
   "ci95": [
    0.5020025860174634,
    0.6905987142287374
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    1
   ]
  },
  {
   "task": "soccer-v3",
   "sigma": 1.5,
   "successes": 29,
   "rollouts": 100,
   "p": 0.29,
   "ci95": [
    0.21014835693844297,
    0.3853889125445355
   ],
   "outcomes": [
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "coffee-push-v3",
   "sigma": 0.3,
   "successes": 59,
   "rollouts": 100,
   "p": 0.59,
   "ci95": [
    0.49201429976362127,
    0.6813268704579594
   ],
   "outcomes": [
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0
   ]
  },
  {
   "task": "coffee-push-v3",
   "sigma": 0.8,
   "successes": 32,
   "rollouts": 100,
   "p": 0.32,
   "ci95": [
    0.23669147266401064,
    0.4166261868928281
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0
   ]
  },
  {
   "task": "coffee-push-v3",
   "sigma": 1.5,
   "successes": 11,
   "rollouts": 100,
   "p": 0.11,
   "ci95": [
    0.06254196329419368,
    0.18631296574562342
   ],
   "outcomes": [
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0
   ]
  },
  {
   "task": "sweep-into-v3",
   "sigma": 0.3,
   "successes": 33,
   "rollouts": 100,
   "p": 0.33,
   "ci95": [
    0.24563122674486615,
    0.4269465628365927
   ],
   "outcomes": [
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0
   ]
  },
  {
   "task": "sweep-into-v3",
   "sigma": 0.8,
   "successes": 21,
   "rollouts": 100,
   "p": 0.21,
   "ci95": [
    0.14165654017044627,
    0.2997996891155716
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "sweep-into-v3",
   "sigma": 1.5,
   "successes": 26,
   "rollouts": 100,
   "p": 0.26,
   "ci95": [
    0.18404698413091963,
    0.3537098952781987
   ],
   "outcomes": [
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "drawer-open-v3",
   "sigma": 0.3,
   "successes": 95,
   "rollouts": 100,
   "p": 0.95,
   "ci95": [
    0.8882495301258955,
    0.9784563209820079
   ],
   "outcomes": [
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "drawer-open-v3",
   "sigma": 0.8,
   "successes": 38,
   "rollouts": 100,
   "p": 0.38,
   "ci95": [
    0.2909759918806736,
    0.4779024478238856
   ],
   "outcomes": [
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0
   ]
  },
  {
   "task": "drawer-open-v3",
   "sigma": 1.5,
   "successes": 12,
   "rollouts": 100,
   "p": 0.12,
   "ci95": [
    0.06999406407369686,
    0.19812099499074032
   ],
   "outcomes": [
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0
   ]
  },
  {
   "task": "reach-v3",
   "sigma": 0.3,
   "successes": 49,
   "rollouts": 100,
   "p": 0.49,
   "ci95": [
    0.3942199885796587,
    0.5865198813957212
   ],
   "outcomes": [
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1
   ]
  },
  {
   "task": "reach-v3",
   "sigma": 0.8,
   "successes": 15,
   "rollouts": 100,
   "p": 0.15,
   "ci95": [
    0.09305985248417561,
    0.23283559665412182
   ],
   "outcomes": [
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0
   ]
  },
  {
   "task": "reach-v3",
   "sigma": 1.5,
   "successes": 5,
   "rollouts": 100,
   "p": 0.05,
   "ci95": [
    0.021543679017992173,
    0.1117504698741045
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "push-v3",
   "sigma": 0.3,
   "successes": 35,
   "rollouts": 100,
   "p": 0.35,
   "ci95": [
    0.2636424818146032,
    0.4474555678160957
   ],
   "outcomes": [
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0
   ]
  },
  {
   "task": "push-v3",
   "sigma": 0.8,
   "successes": 7,
   "rollouts": 100,
   "p": 0.07,
   "ci95": [
    0.0343192608798127,
    0.13749514806152413
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "push-v3",
   "sigma": 1.5,
   "successes": 0,
   "rollouts": 100,
   "p": 0.0,
   "ci95": [
    0.0,
    0.03699349876899627
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "pick-place-v3",
   "sigma": 0.3,
   "successes": 55,
   "rollouts": 100,
   "p": 0.55,
   "ci95": [
    0.4524460292190725,
    0.643854620904028
   ],
   "outcomes": [
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1
   ]
  },
  {
   "task": "pick-place-v3",
   "sigma": 0.8,
   "successes": 13,
   "rollouts": 100,
   "p": 0.13,
   "ci95": [
    0.07757167395660301,
    0.20980351513245427
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "pick-place-v3",
   "sigma": 1.5,
   "successes": 2,
   "rollouts": 100,
   "p": 0.02,
   "ci95": [
    0.0055019675006616475,
    0.07001179131757478
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "plate-slide-v3",
   "sigma": 0.3,
   "successes": 100,
   "rollouts": 100,
   "p": 1.0,
   "ci95": [
    0.9630065012310038,
    1.0
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "plate-slide-v3",
   "sigma": 0.8,
   "successes": 68,
   "rollouts": 100,
   "p": 0.68,
   "ci95": [
    0.583373813107172,
    0.7633085273359894
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1
   ]
  },
  {
   "task": "plate-slide-v3",
   "sigma": 1.5,
   "successes": 36,
   "rollouts": 100,
   "p": 0.36,
   "ci95": [
    0.2727122033881362,
    0.45764597626718284
   ],
   "outcomes": [
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0
   ]
  },
  {
   "task": "faucet-open-v3",
   "sigma": 0.3,
   "successes": 7,
   "rollouts": 100,
   "p": 0.07,
   "ci95": [
    0.0343192608798127,
    0.13749514806152413
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "faucet-open-v3",
   "sigma": 0.8,
   "successes": 44,
   "rollouts": 100,
   "p": 0.44,
   "ci95": [
    0.3467202656095169,
    0.5377189542427627
   ],
   "outcomes": [
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1
   ]
  },
  {
   "task": "faucet-open-v3",
   "sigma": 1.5,
   "successes": 77,
   "rollouts": 100,
   "p": 0.77,
   "ci95": [
    0.6784561689907251,
    0.841567341674017
   ],
   "outcomes": [
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "handle-press-v3",
   "sigma": 0.3,
   "successes": 0,
   "rollouts": 100,
   "p": 0.0,
   "ci95": [
    0.0,
    0.03699349876899627
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "handle-press-v3",
   "sigma": 0.8,
   "successes": 19,
   "rollouts": 100,
   "p": 0.19,
   "ci95": [
    0.12514751467908536,
    0.27778845455769235
   ],
   "outcomes": [
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0
   ]
  },
  {
   "task": "handle-press-v3",
   "sigma": 1.5,
   "successes": 25,
   "rollouts": 100,
   "p": 0.25,
   "ci95": [
    0.17545211311910353,
    0.34304463626539466
   ],
   "outcomes": [
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    0,
    0,
    1
   ]
  },
  {
   "task": "lever-pull-v3",
   "sigma": 0.3,
   "successes": 0,
   "rollouts": 100,
   "p": 0.0,
   "ci95": [
    0.0,
    0.03699349876899627
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "lever-pull-v3",
   "sigma": 0.8,
   "successes": 4,
   "rollouts": 100,
   "p": 0.04,
   "ci95": [
    0.01566330388275538,
    0.09837071498472119
   ],
   "outcomes": [
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "lever-pull-v3",
   "sigma": 1.5,
   "successes": 1,
   "rollouts": 100,
   "p": 0.01,
   "ci95": [
    0.0017674320446096452,
    0.05448619674900669
   ],
   "outcomes": [
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0,
    0
   ]
  },
  {
   "task": "drawer-close-v3",
   "sigma": 0.3,
   "successes": 100,
   "rollouts": 100,
   "p": 1.0,
   "ci95": [
    0.9630065012310038,
    1.0
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "drawer-close-v3",
   "sigma": 0.8,
   "successes": 98,
   "rollouts": 100,
   "p": 0.98,
   "ci95": [
    0.9299882086824253,
    0.9944980324993383
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "drawer-close-v3",
   "sigma": 1.5,
   "successes": 97,
   "rollouts": 100,
   "p": 0.97,
   "ci95": [
    0.9154806351014251,
    0.9897454760557185
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "door-close-v3",
   "sigma": 0.3,
   "successes": 99,
   "rollouts": 100,
   "p": 0.99,
   "ci95": [
    0.9455138032509935,
    0.9982325679553905
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "door-close-v3",
   "sigma": 0.8,
   "successes": 93,
   "rollouts": 100,
   "p": 0.93,
   "ci95": [
    0.8625048519384759,
    0.9656807391201874
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1
   ]
  },
  {
   "task": "door-close-v3",
   "sigma": 1.5,
   "successes": 98,
   "rollouts": 100,
   "p": 0.98,
   "ci95": [
    0.9299882086824253,
    0.9944980324993383
   ],
   "outcomes": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1
   ]
  }
 ]
}

In [ ]:
import os, json, time, traceback, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import numpy as np, torch
from rber.domain import load_library, build_world
from rber import llm

SEEDS = [0, 1, 2, 3]
METHODS = ['VERIF', 'STEP', 'EXEC', 'RBER', 'RBER-NL']
OUT = "/kaggle/working/llm_results_sessionA" if os.path.exists("/kaggle") else "llm_results_sessionA"
os.makedirs(OUT, exist_ok=True)
LOG = open(os.path.join(OUT, "log.txt"), "a")
def log(*a):
    s = " ".join(str(x) for x in a); print(s, flush=True); LOG.write(s + "\n"); LOG.flush()

lib = load_library("skills_metaworld.json")
W = build_world(lib, K=3, n_train=24, n_held=12, seed=0)
log("=== session A", time.ctime(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
log("skills:", len(lib.p), "| optimum train/held:", np.mean([W.optimum(t) for t in W.train]), np.mean([W.optimum(t) for t in W.held]))

In [ ]:
for s in SEEDS:
    for m in METHODS:
        try:
            llm.train(W, m, s, OUT, lr=3e-6, n_updates=250, tasks_per_update=4, G=8, eval_every=25, log=log)
        except Exception:
            log("!! run", m, s, "crashed (continuing):\n" + traceback.format_exc())
            import gc; gc.collect(); torch.cuda.empty_cache()
        log(f"GPU memory after run: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
zp = shutil.make_archive(OUT, "zip", OUT)
log("Download:", zp)